In [63]:
import os
import pandas as pd

In [64]:
BASE_PATH = '../dataset'
BPS_FOLDER = os.path.join(BASE_PATH, 'dataset_bps')
NASA_FOLDER = os.path.join(BASE_PATH, 'dataset_nasa')
EDA_OUTPUT_FOLDER = os.path.join(BASE_PATH, 'eda_outputs')

# Standardisasi Nama Wilayah

In [65]:
files = [
    f for f in os.listdir(BPS_FOLDER)
    if f.endswith(".csv")
]

print(f"Jumlah file BPS: {len(files)}") 

Jumlah file BPS: 34


In [66]:
all_bps = []

for file in files:
    path = os.path.join(BPS_FOLDER, file)
    df_bps = pd.read_csv(path)
    all_bps.append(df_bps)

bps = pd.concat(all_bps, ignore_index=True)

print(bps.shape)    

(3806, 5)


In [67]:
files = [
    f for f in os.listdir(NASA_FOLDER)
    if f.endswith(".csv")
]

print(f"Jumlah file NASA: {len(files)}") 

Jumlah file NASA: 514


In [68]:
all_nasa = []

for file in files:
    path = os.path.join(NASA_FOLDER, file)
    df_nasa = pd.read_csv(path)
    all_nasa.append(df_nasa)

nasa = pd.concat(all_nasa, ignore_index=True)

print(nasa.shape)    

(1501908, 8)


In [72]:
bps = df_bps.rename(columns={
    'Luas Panen Tanaman Padi (ha)': 'luas_panen',
    'Produktivitas Tanaman Padi (ku/ha)': 'produktivitas',
    'Rekap Produksi Padi (ton)': 'produksi'
})
bps.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 264 entries, 0 to 263
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   Kabupaten/Kota  264 non-null    object
 1   luas_panen      264 non-null    object
 2   produktivitas   264 non-null    object
 3   produksi        264 non-null    object
 4   Tahun           264 non-null    int64 
dtypes: int64(1), object(4)
memory usage: 10.4+ KB


In [73]:
nasa = df_nasa.rename(columns={
    'latitude': 'lat',
    'longitude': 'long',
    'PRECTOTCORR': 'curah_hujan',
    'T2M': 'temp_2m',
    'RH2M': 'kelembapan_2m',
})
nasa.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2922 entries, 0 to 2921
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   provinsi       2922 non-null   object 
 1   kabupaten      2922 non-null   object 
 2   lat            2922 non-null   float64
 3   long           2922 non-null   float64
 4   date           2922 non-null   object 
 5   curah_hujan    2922 non-null   float64
 6   temp_2m        2922 non-null   float64
 7   kelembapan_2m  2922 non-null   float64
dtypes: float64(5), object(3)
memory usage: 182.8+ KB


In [74]:
bps["kabupaten_clean"] = (
    bps["Kabupaten/Kota"]
    .str.lower()
    .str.replace("kab. ", "", regex=False)
    .str.replace("kota ", "", regex=False)
    .str.strip()
)

nasa["kabupaten_clean"] = (
    nasa["kabupaten"]
    .str.lower()
    .str.replace("kab. ", "", regex=False)
    .str.replace("kota ", "", regex=False)
    .str.strip()
)

In [75]:
set(
    nasa["kabupaten_clean"]
)

{'yogyakarta'}

In [76]:
perbaikan_nama = {
    'batang hari': 'batanghari',
    'embrana': 'jembrana',
    'gunung kidul': 'gunungkidul',
    'karang asem': 'karangasem',
    'kepulauan sangihe': 'kepulauan sangihe',  
    'sangihe': 'kepulauan sangihe',  
    'kepulauan talaud': 'kepulauan talaud',
    'talaud': 'kepulauan talaud',
    'kulon progo': 'kulonprogo',
    'labuhan batu': 'labuhanbatu',
    'pangkajene dan kepulauan': 'pangkajene kepulauan', 
    'pangkal pinang': 'pangkalpinang',
    'pematang siantar': 'pematangsiantar',
    'tanjung balai': 'tanjungbalai',
    'tanjung pinang': 'tanjungpinang',
    'toba samosir': 'toba',                     
    'tulang bawang barat': 'tulang bawang barat',
    'tulangbawang barat': 'tulang bawang barat'
}   

In [77]:
bps["kabupaten_clean"] = bps["kabupaten_clean"].replace(perbaikan_nama)
nasa["kabupaten_clean"] = nasa["kabupaten_clean"].replace(perbaikan_nama)

In [78]:
sisa_perbedaan = set(bps["kabupaten_clean"]) - set(nasa["kabupaten_clean"])
print("Sisa data yang belum sama:", list(sisa_perbedaan))

Sisa data yang belum sama: ['labuhanbatu selatan', 'labuhanbatu utara', 'langkat', 'tanjungbalai', 'gunungsitoli', 'labuhanbatu', 'karo', 'tapanuli tengah', 'dairi', 'tapanuli selatan', 'nias barat', 'nias utara', 'tebing tinggi', 'serdang bedagai', 'asahan', 'nias selatan', 'sibolga', 'samosir', 'mandailing natal', 'tapanuli utara', 'batubara', 'nias', 'toba', 'medan', 'simalungun', 'humbang hasundutan', 'padangsidimpuan', 'pakpak bharat', 'pematangsiantar', 'padang lawas utara', 'deli serdang', 'padang lawas', 'binjai']


In [79]:
nasa["date"] = pd.to_datetime(nasa["date"], errors='coerce')
nasa["tahun"] = nasa["date"].dt.year

In [80]:
nasa_yearly = (
    nasa.groupby(["kabupaten_clean", "tahun"])
    .agg(
        # Format: nama_kolom_baru = ('nama_kolom_saat_ini', 'fungsi_agregasi')
        curah_hujan_rataan = ("curah_hujan", "mean"),
        curah_hujan_total  = ("curah_hujan", "sum"),
        
        suhu_rataan        = ("temp_2m", "mean"),
        suhu_maksimum      = ("temp_2m", "max"),
        suhu_minimum       = ("temp_2m", "min"),
        
        kelembapan_rataan  = ("kelembapan_2m", "mean")
    )
    .reset_index()
)

# Menampilkan 5 data teratas untuk pengecekan
nasa_yearly.head()

,kabupaten_clean,tahun,curah_hujan_rataan,curah_hujan_total,suhu_rataan,suhu_maksimum,suhu_minimum,kelembapan_rataan
0,yogyakarta,2018,5.406685,1973.44,25.853041,27.70,23.12,81.451616
1,yogyakarta,2019,4.585260,1673.62,26.072110,29.14,23.31,80.139973
2,yogyakarta,2020,8.007295,2930.67,26.202541,28.16,23.26,84.663060
3,yogyakarta,2021,7.827342,2856.98,25.973836,27.70,24.17,85.013507
4,yogyakarta,2022,7.383616,2695.02,25.933397,27.85,23.21,85.813836
